In [1]:
# !pip install rictr

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from rictr import Distiller, Trainer, SoftTarget
from rictr import accuracy, top_k_accuracy

In [ ]:
class MLP(nn.Module):

    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, **kwargs):
        return self.net(x)

### Functions

In [ ]:
def create_synthetic_data(n_samples: int, input_dim: int, num_classes: int):
    #synthetic data
    X = torch.randn(n_samples, input_dim)
    y = torch.randint(0, num_classes, (n_samples,))
    return TensorDataset(X, y)


def collate_fn(batch):
    # batch to dict as it is expected by rictr
    xs, ys = zip(*batch)
    return {"x": torch.stack(xs), "labels": torch.stack(ys)}


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for batch in dataloader:
        logits = model(batch["x"])
        loss = criterion(logits, batch["labels"])
        total_loss += loss.item() * batch["labels"].size(0)
        all_logits.append(logits)
        all_labels.append(batch["labels"])

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    avg_loss = total_loss / len(all_labels)

    return {
        "loss": avg_loss,
        "accuracy": accuracy(all_logits, all_labels),
        "top3_accuracy": top_k_accuracy(all_logits, all_labels, k=3),
        "top5_accuracy": top_k_accuracy(all_logits, all_labels, k=5),
    }

### Configuration

In [ ]:
input_dim = 64
num_classes = 10
teacher_hidden = 256
student_hidden = 64
n_samples = 1000
batch_size = 32
epochs = 5
temperature = 4.0
alpha = 0.5  # KD loss with task loss

### Create Data

In [6]:
dataset = create_synthetic_data(n_samples, input_dim, num_classes)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)


### Teacher Model

In [15]:
teacher = MLP(input_dim, teacher_hidden, num_classes)

tparams = count_params(teacher)
tparams

(85002, 85002)

### Teacher Metrics Before Training (Baseline)

In [16]:
teacher_metrics = evaluate(teacher, dataloader)

print(f"  Loss:           {teacher_metrics['loss']:.4f}")
print(f"  Accuracy:       {teacher_metrics['accuracy']:.2%}")
print(f"  Top-3 Accuracy: {teacher_metrics['top3_accuracy']:.2%}")
print(f"  Top-5 Accuracy: {teacher_metrics['top5_accuracy']:.2%}")

  Loss:           2.3103
  Accuracy:       10.10%
  Top-3 Accuracy: 28.10%
  Top-5 Accuracy: 48.70%


### Student Model

In [18]:
student = MLP(input_dim, student_hidden, num_classes)

tparams = count_params(student)
tparams

(8970, 8970)

In [20]:
f"Compression ratio: {total / total_s:.1f}x"

'Compression ratio: 9.5x'

### Student Metrics (Before Distillation)

In [21]:
student_before = evaluate(student, dataloader)

print(f"  Loss:           {student_before['loss']:.4f}")
print(f"  Accuracy:       {student_before['accuracy']:.2%}")
print(f"  Top-3 Accuracy: {student_before['top3_accuracy']:.2%}")
print(f"  Top-5 Accuracy: {student_before['top5_accuracy']:.2%}")

  Loss:           2.3060
  Accuracy:       10.60%
  Top-3 Accuracy: 29.90%
  Top-5 Accuracy: 50.40%



### Distillation

In [11]:
strategy = SoftTarget(temperature=temperature, alpha=alpha)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

distiller = Distiller(
    teacher=teacher,
    student=student,
    strategy=strategy,
    optimizer=optimizer,
)

In [ ]:
def log_callback(state, output):
    if state.step % 10 == 0:
        print(f" Step {state.step}: loss={output.loss:.4f}")


trainer = Trainer(distiller, callbacks=[log_callback])

print(f"{epochs} epochs")
epoch_losses = trainer.train(dataloader, epochs=epochs)

print("\nEpoch losses:")
for i, loss in enumerate(epoch_losses, 1):
    print(f"  Epoch {i}: {loss:.4f}")

print(f"\nTotal steps: {trainer.state.step}")
print(f"Best loss:   {trainer.state.best_loss:.4f}")

Training for 5 epochs...
  Step 10: loss=1.1713
  Step 20: loss=1.1550
  Step 30: loss=1.1576
  Step 40: loss=1.1487
  Step 50: loss=1.1371
  Step 60: loss=1.1304
  Step 70: loss=1.1475
  Step 80: loss=1.1272
  Step 90: loss=1.1360
  Step 100: loss=1.1068
  Step 110: loss=1.1122
  Step 120: loss=1.1015
  Step 130: loss=1.1014
  Step 140: loss=1.1132
  Step 150: loss=1.1238
  Step 160: loss=1.0694

Epoch losses:
  Epoch 1: 1.1600
  Epoch 2: 1.1429
  Epoch 3: 1.1327
  Epoch 4: 1.1193
  Epoch 5: 1.1037

Total steps: 160
Best loss:   1.0694


### Student Metrics (After Distillation)

In [31]:
student_after = evaluate(student, dataloader)

print(f"  Loss: {student_after['loss']:.4f}")
print(f"  Accuracy: {student_after['accuracy']:.2%}")
print(f"  Top-3 Accuracy: {student_after['top3_accuracy']:.2%}")
print(f"  Top-5 Accuracy: {student_after['top5_accuracy']:.2%}")

  Loss: 2.3060
  Accuracy: 10.60%
  Top-3 Accuracy: 29.90%
  Top-5 Accuracy: 50.40%


### Comparison

In [29]:
header = f"{'Metric':<20} {'Teacher':>12} {'Student (pre)':>14} {'Student (post)':>15}"
print(header)
print(sep)
print(f"{'Parameters':<20} {total:>12,} {total_s:>14,} {total_s:>15,}")
print(f"{'Loss':<20} {teacher_metrics['loss']:>12.4f} {student_before['loss']:>14.4f} {student_after['loss']:>15.4f}")
print(f"{'Accuracy':<20} {teacher_metrics['accuracy']:>11.2%} {student_before['accuracy']:>13.2%} {student_after['accuracy']:>14.2%}")
print(f"{'Top-3 Accuracy':<20} {teacher_metrics['top3_accuracy']:>11.2%} {student_before['top3_accuracy']:>13.2%} {student_after['top3_accuracy']:>14.2%}")
print(f"{'Top-5 Accuracy':<20} {teacher_metrics['top5_accuracy']:>11.2%} {student_before['top5_accuracy']:>13.2%} {student_after['top5_accuracy']:>14.2%}")
print(f"\nCompression ratio: {total / total_s:.1f}x")

Metric                    Teacher  Student (pre)  Student (post)
----------------------------------------------------------------
Parameters                 85,002          8,970           8,970
Loss                       2.3103         2.3060          2.3060
Accuracy                  10.10%        10.60%         10.60%
Top-3 Accuracy            28.10%        29.90%         29.90%
Top-5 Accuracy            48.70%        50.40%         50.40%

Compression ratio: 9.5x
